8mins 30.7 secs

## Libraries

In [43]:
import numpy as np
import pandas as pd
import time
import random

from sklearn.model_selection import ParameterGrid

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

In [44]:
print("Torch version:", torch.__version__)
print("CUDA (in torch):", torch.version.cuda)
print("cuda.is_available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

Torch version: 2.10.0.dev20251203+cu128
CUDA (in torch): 12.8
cuda.is_available: True
device count: 1
device: NVIDIA GeForce RTX 5070 Ti
capability: (12, 0)


## Config

In [45]:
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

# Tuning window (no test here; just 2007-04 → 2022-03)
TUNE_START_DATE = pd.Timestamp("2007-04-01")
TUNE_END_DATE   = pd.Timestamp("2022-03-31")

# Rolling CV: 10 years train, 1 year val (in months)
ROLLING_TRAIN_WINDOW = 90   # 90 months = 7.5 years
ROLLING_VAL_WINDOW   = 18    # 18 months = 1.5 years

# Hyperparameter grid for T-GCN
param_grid = list(ParameterGrid({
    "WINDOW":      [12, 18],     # sequence length
    "HIDDEN_DIM":  [32, 64, 128],
    "DROPOUT":     [0.0, 0.3],
    "LR":          [1e-3, 5e-4],
    "WEIGHT_DECAY":[0.0, 1e-4],
}))

print(f"Number of hyperparameter configs: {len(param_grid)}")

BATCH_SIZE    = 32
MAX_EPOCHS    = 80
PATIENCE      = 6
MAX_GRAD_NORM = 5.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# feature lists
continuous_cols = [
    "AverageNeighbourPrice","local_I","area_km2","centroid_x","centroid_y",
    "CoL_distance_km","LA_FE","sdlt_perc_threshold","dwelling_stock",
    "population","ashe_weekly","base_rate","claimant_count_prop",
    "planning_decisions_per_1000","planning_granted_prop",
    "rail_station_entry_exit","GDP","CPIH",
]

categorical_cols = [
    "LMIQuadrant__2","LMIQuadrant__3","LMIQuadrant__4",
    "Region_East of England","Region_London","Region_North East",
    "Region_North West","Region_South East","Region_South West",
    "Region_West Midlands","Region_Yorkshire and The Humber"
]

base_feature_cols = continuous_cols + categorical_cols

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Number of hyperparameter configs: 48
Using device: cuda


## Metric functions

In [46]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(
        100.0 * np.mean(
            2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)
        )
    )

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)

    if len(y_train) <= m:
        return np.nan

    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)


## Load data

In [47]:
df = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

# restrict to tuning period
mask_tune = (df[TIME_COL] >= TUNE_START_DATE) & (df[TIME_COL] <= TUNE_END_DATE)
df = df.loc[mask_tune].copy()

# ensure 1 row per (Date, AreaCode)
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)

print(f"Tuning period: {dates[0].date()} → {dates[-1].date()}")
print("Total months:", T_total)
print("Number of LAs:", N)

full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

feature_cols = base_feature_cols.copy()

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# ffill/bfill features + target within each LA
df_panel[feature_cols + [TARGET_COL]] = (
    df_panel[feature_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

# ---- add price lags 1 & 12 ----
df_panel["price_lag1"] = (
    df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(1)
)
df_panel["price_lag12"] = (
    df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(12)
)

df_panel[["price_lag1", "price_lag12"]] = (
    df_panel[["price_lag1", "price_lag12"]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

lag_price_cols = ["price_lag1", "price_lag12"]
feature_cols = feature_cols + lag_price_cols

# last-resort fill for any remaining NaNs
missing_total = df_panel[feature_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after lag creation. Filling with column means.")
    col_means = df_panel[feature_cols + [TARGET_COL]].mean()
    df_panel[feature_cols + [TARGET_COL]] = df_panel[feature_cols + [TARGET_COL]].fillna(col_means)

print("NaNs after panel completion:",
      df_panel[feature_cols + [TARGET_COL]].isna().sum().sum())

F = len(feature_cols)

X_all = (
    df_panel[feature_cols]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N, F)
)
y_all = (
    df_panel[TARGET_COL]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N)
)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

y_all_orig = y_all.copy()

Tuning period: 2007-04-01 → 2022-03-01
Total months: 180
Number of LAs: 294
⚠ 1 NaNs after lag creation. Filling with column means.
NaNs after panel completion: 0
X_all shape: (180, 294, 31)
y_all shape: (180, 294)


## Node order and adjacency

In [48]:
# # use K=8 neighbours to avoid low degree
# nbrs = NearestNeighbors(n_neighbors=8).fit(centroids)
# _, idx = nbrs.kneighbors(centroids)

# A = np.zeros((N, N), dtype=np.float32)
# for i in range(N):
#     for j in idx[i][1:]:   # skip self
#         A[i, j] = 1.0
#         A[j, i] = 1.0

# # add self-loops and normalise
# A = A + np.eye(N, dtype=np.float32)
# deg = A.sum(axis=1)
# print("Min/Max degree after KNN+I:", deg.min(), deg.max())

# D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + 1e-8))
# A_hat_np = D_inv_sqrt @ A @ D_inv_sqrt
# A_hat = torch.tensor(A_hat_np, dtype=torch.float32, device=DEVICE)

# print("A_hat shape:", A_hat.shape)

## Correlation-Based KNN adjacency

In [49]:
y_corr = y_all  # [T_total, N]

corr = np.corrcoef(y_corr.T)   # [N, N]
corr = np.nan_to_num(corr, nan=0.0)
np.fill_diagonal(corr, 0.0)

K = 8  # neighbours per node
A = np.zeros((N, N), dtype=np.float32)
for i in range(N):
    nbr_idx = np.argsort(-np.abs(corr[i]))[:K]
    for j in nbr_idx:
        A[i, j] = 1.0
        A[j, i] = 1.0

A = A + np.eye(N, dtype=np.float32)
deg = A.sum(axis=1)
print("Min/Max degree (corr graph):", deg.min(), deg.max())

D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + 1e-8))
A_hat_np = D_inv_sqrt @ A @ D_inv_sqrt
A_hat = torch.tensor(A_hat_np, dtype=torch.float32, device=DEVICE)
print("A_hat shape:", A_hat.shape)

Min/Max degree (corr graph): 9.0 43.0
A_hat shape: torch.Size([294, 294])


## Rolling origin folds (time index space)

In [50]:
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW   # first index where a val window can start

while True:
    train_end_idx = start_idx            # exclusive
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > T_total:
        break

    train_start_idx = train_end_idx - ROLLING_TRAIN_WINDOW

    train_start_date = dates[train_start_idx]
    train_end_date   = dates[train_end_idx - 1]
    val_start_date   = dates[val_start_idx]
    val_end_date     = dates[val_end_idx - 1]

    fold_specs.append(
        (train_start_idx, train_end_idx, val_start_idx, val_end_idx,
         train_start_date, train_end_date, val_start_date, val_end_date)
    )

    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")
for i, (tr_s, tr_e, v_s, v_e, ts, te, vs, ve) in enumerate(fold_specs, start=1):
    print(f"  Fold {i}: Train {ts:%Y-%m}–{te:%Y-%m}, Val {vs:%Y-%m}–{ve:%Y-%m}")


Number of folds: 5
  Fold 1: Train 2007-04–2014-09, Val 2014-10–2016-03
  Fold 2: Train 2008-10–2016-03, Val 2016-04–2017-09
  Fold 3: Train 2010-04–2017-09, Val 2017-10–2019-03
  Fold 4: Train 2011-10–2019-03, Val 2019-04–2020-09
  Fold 5: Train 2013-04–2020-09, Val 2020-10–2022-03


## Dataset class (window will vary per config)

In [51]:
class SpatioTemporalDataset(Dataset):
    """
    Each sample is (X_seq, y_t):
      - X_seq: [window, N, F]
      - y_t: [N]
    """
    def __init__(self, X, y, window):
        self.X = X
        self.y = y
        self.window = window
        self.T, self.N, self.F = X.shape

        self.indices = list(range(window, self.T))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        X_seq = self.X[t - self.window:t]  # [window, N, F]
        y_t   = self.y[t]                 # [N]
        return (
            torch.tensor(X_seq, dtype=torch.float32),
            torch.tensor(y_t,   dtype=torch.float32),
        )

## Model

In [52]:
class GraphConv(nn.Module):
    def __init__(self, in_feats, out_feats):
        super().__init__()
        self.linear = nn.Linear(in_feats, out_feats)

    def forward(self, X, A_hat):
        # X: [B, N, F], A_hat: [N, N]
        return self.linear(torch.einsum("ij,bjf->bif", A_hat, X))

class TGCNCell(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gc_zr = GraphConv(in_feats + hidden_dim, 2 * hidden_dim)
        self.gc_h  = GraphConv(in_feats + hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X_t, H_prev, A_hat):
        if H_prev is None:
            H_prev = torch.zeros(
                X_t.size(0), X_t.size(1), self.hidden_dim, device=X_t.device
            )
        XH = torch.cat([X_t, H_prev], dim=-1)

        ZR = torch.sigmoid(self.gc_zr(XH, A_hat))
        Z, R = torch.chunk(ZR, 2, dim=-1)

        XH_candidate = torch.cat([X_t, R * H_prev], dim=-1)
        H_tilde = torch.tanh(self.gc_h(XH_candidate, A_hat))

        H_new = (1 - Z) * H_prev + Z * H_tilde
        H_new = self.dropout(H_new)
        return H_new

class TGCN(nn.Module):
    def __init__(self, num_nodes, in_feats, hidden_dim, dropout=0.0):
        super().__init__()
        self.cell = TGCNCell(in_feats, hidden_dim, dropout=dropout)
        self.out  = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_hat):
        """
        X_seq: [B, T, N, F]
        """
        B, T_seq, N_nodes, F_in = X_seq.shape
        H = None
        for t in range(T_seq):
            X_t = X_seq[:, t]    # [B, N, F]
            H   = self.cell(X_t, H, A_hat)
        y_hat = self.out(H).squeeze(-1)  # [B, N]
        return y_hat


## Train and Eval with early stopping

In [53]:
results = []

for cfg_id, params in enumerate(param_grid, start=1):
    WINDOW      = params["WINDOW"]
    HIDDEN_DIM  = params["HIDDEN_DIM"]
    DROPOUT     = params["DROPOUT"]
    LR          = params["LR"]
    WD          = params["WEIGHT_DECAY"]

    print(f"\n=== Config {cfg_id}/{len(param_grid)} ===")
    print(params)

    fold_mae_list   = []
    fold_rmse_list  = []
    fold_smape_list = []
    fold_mase_list  = []

    fold_count = 0

    for fold_no, (train_start_idx, train_end_idx,
                  val_start_idx, val_end_idx,
                  train_start_date, train_end_date,
                  val_start_date, val_end_date) in enumerate(fold_specs, start=1):

        print(f"  Fold {fold_no}: Train {train_start_date:%Y-%m}–{train_end_date:%Y-%m}, "
              f"Val {val_start_date:%Y-%m}–{val_end_date:%Y-%m}")

        # -------- LEAK-FREE SCALING FOR THIS FOLD ONLY --------
        X_train = X_all[train_start_idx:train_end_idx]   # [T_tr, N, F]
        X_val   = X_all[val_start_idx:val_end_idx]       # [T_val, N, F]

        y_train = y_all[train_start_idx:train_end_idx]   # [T_tr, N]
        y_val   = y_all[val_start_idx:val_end_idx]       # [T_val, N]

        # fit scalers only on training window
        x_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(
            X_train.reshape(-1, F)
        ).reshape(X_train.shape)

        X_val_scaled = x_scaler.transform(
            X_val.reshape(-1, F)
        ).reshape(X_val.shape)

        y_scaler = RobustScaler()
        y_train_scaled = y_scaler.fit_transform(
            y_train.reshape(-1, 1)
        ).reshape(y_train.shape)

        y_val_scaled = y_scaler.transform(
            y_val.reshape(-1, 1)
        ).reshape(y_val.shape)

        y_scale_factor = float(y_scaler.scale_[0])

        # build datasets using local (train/val) segments
        if X_train_scaled.shape[0] <= WINDOW or X_val_scaled.shape[0] <= WINDOW:
            print("    (skip fold: not enough time steps after WINDOW constraint)")
            continue

        train_ds = SpatioTemporalDataset(
            X_train_scaled, y_train_scaled, window=WINDOW
        )
        val_ds = SpatioTemporalDataset(
            X_val_scaled, y_val_scaled, window=WINDOW
        )

        if len(train_ds) == 0 or len(val_ds) == 0:
            print("    (skip fold: empty dataset)")
            continue

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

        model = TGCN(
            num_nodes=N,
            in_feats=F,
            hidden_dim=HIDDEN_DIM,
            dropout=DROPOUT
        ).to(DEVICE)

        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
        loss_fn = nn.MSELoss()

        best_val_mse = np.inf
        best_epoch = -1
        epochs_no_improve = 0
        best_state = None

        # --------------- TRAIN + EARLY STOP ---------------
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            train_losses = []

            for X_seq, y_t in train_loader:
                X_seq = X_seq.to(DEVICE)  # [B, T, N, F]
                y_t   = y_t.to(DEVICE)    # [B, N]

                optimizer.zero_grad()
                y_hat = model(X_seq, A_hat)  # [B, N]
                loss  = loss_fn(y_hat, y_t)

                if not torch.isfinite(loss):
                    print(f"    ⚠ Non-finite loss at epoch {epoch}. Skipping fold.")
                    train_losses = []
                    break

                loss.backward()
                clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
                optimizer.step()
                train_losses.append(loss.item())

            if not train_losses:
                break

            # validation
            model.eval()
            val_losses = []
            with torch.no_grad():
                for X_seq, y_t in val_loader:
                    X_seq = X_seq.to(DEVICE)
                    y_t   = y_t.to(DEVICE)
                    y_hat = model(X_seq, A_hat)
                    vloss = loss_fn(y_hat, y_t)
                    if torch.isfinite(vloss):
                        val_losses.append(vloss.item())

            if not val_losses:
                print("    ⚠ All val losses non-finite. Skipping fold.")
                break

            val_mse = float(np.mean(val_losses))
            val_rmse_orig = np.sqrt(val_mse) * y_scale_factor

            print(f"    Epoch {epoch:03d} | "
                  f"train MSE={np.mean(train_losses):.4f} | "
                  f"val MSE={val_mse:.4f} | "
                  f"val RMSE(£)≈{val_rmse_orig:,.1f}")

            if val_mse + 1e-6 < best_val_mse:
                best_val_mse = val_mse
                best_epoch   = epoch
                epochs_no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= PATIENCE:
                    print(f"    Early stopping at epoch {epoch}")
                    break

        if best_epoch == -1 or best_state is None:
            print("    ❌ Fold failed (no valid epoch). Skipping fold.")
            continue

        # restore best state
        model.load_state_dict(best_state)

        # --------------- FINAL VAL METRICS FOR THIS FOLD ---------------
        model.eval()
        y_true_val_scaled = []
        y_pred_val_scaled = []

        with torch.no_grad():
            for X_seq, y_t in val_loader:
                X_seq = X_seq.to(DEVICE)
                y_t   = y_t.to(DEVICE)
                y_hat = model(X_seq, A_hat)
                y_true_val_scaled.append(y_t.cpu().numpy())   # [B, N]
                y_pred_val_scaled.append(y_hat.cpu().numpy()) # [B, N]

        y_true_val_scaled = np.concatenate(y_true_val_scaled, axis=0).reshape(-1, 1)
        y_pred_val_scaled = np.concatenate(y_pred_val_scaled, axis=0).reshape(-1, 1)

        y_true_val_orig = y_scaler.inverse_transform(y_true_val_scaled).ravel()
        y_pred_val_orig = y_scaler.inverse_transform(y_pred_val_scaled).ravel()

        # MASE scaling: use TRAIN window in original scale
        y_train_fold_orig = y_all_orig[train_start_idx:train_end_idx].reshape(-1)

        fold_mae  = mae(y_true_val_orig, y_pred_val_orig)
        fold_rmse = rmse(y_true_val_orig, y_pred_val_orig)
        fold_smape = smape(y_true_val_orig, y_pred_val_orig)
        fold_mase = mase(y_true_val_orig, y_pred_val_orig, y_train_fold_orig, m=12)

        print(f"    Fold {fold_no} MAE(£)={fold_mae:,.1f}, "
              f"RMSE(£)={fold_rmse:,.1f}, sMAPE={fold_smape:.3f}%, MASE={fold_mase:.3f}")

        fold_mae_list.append(fold_mae)
        fold_rmse_list.append(fold_rmse)
        fold_smape_list.append(fold_smape)
        fold_mase_list.append(fold_mase)
        fold_count += 1

    if fold_count == 0:
        print("  ❌ No valid folds for this config. Skipping.")
        continue

    cfg_result = {
        "model_type": "TGCN",
        "WINDOW": WINDOW,
        "HIDDEN_DIM": HIDDEN_DIM,
        "DROPOUT": DROPOUT,
        "LR": LR,
        "WEIGHT_DECAY": WD,
        "folds_used": fold_count,

        "MAE_mean":   float(np.mean(fold_mae_list)),
        "MAE_std":    float(np.std(fold_mae_list)),
        "RMSE_mean":  float(np.mean(fold_rmse_list)),
        "RMSE_std":   float(np.std(fold_rmse_list)),
        "sMAPE_mean": float(np.mean(fold_smape_list)),
        "sMAPE_std":  float(np.std(fold_smape_list)),
        "MASE_mean":  float(np.mean(fold_mase_list)),
        "MASE_std":   float(np.std(fold_mase_list)),
    }
    results.append(cfg_result)


=== Config 1/48 ===
{'DROPOUT': 0.0, 'HIDDEN_DIM': 32, 'LR': 0.001, 'WEIGHT_DECAY': 0.0, 'WINDOW': 12}
  Fold 1: Train 2007-04–2014-09, Val 2014-10–2016-03
    Epoch 001 | train MSE=1.0758 | val MSE=2.8825 | val RMSE(£)≈172,530.0
    Epoch 002 | train MSE=0.8435 | val MSE=2.5010 | val RMSE(£)≈160,708.9
    Epoch 003 | train MSE=0.7018 | val MSE=2.1781 | val RMSE(£)≈149,975.2
    Epoch 004 | train MSE=0.6240 | val MSE=1.9051 | val RMSE(£)≈140,260.0
    Epoch 005 | train MSE=0.5376 | val MSE=1.6718 | val RMSE(£)≈131,391.6
    Epoch 006 | train MSE=0.4783 | val MSE=1.4783 | val RMSE(£)≈123,555.6
    Epoch 007 | train MSE=0.4571 | val MSE=1.3266 | val RMSE(£)≈117,043.5
    Epoch 008 | train MSE=0.4432 | val MSE=1.2064 | val RMSE(£)≈111,617.7
    Epoch 009 | train MSE=0.4138 | val MSE=1.1102 | val RMSE(£)≈107,073.1
    Epoch 010 | train MSE=0.3979 | val MSE=1.0383 | val RMSE(£)≈103,550.1
    Epoch 011 | train MSE=0.3815 | val MSE=0.9839 | val RMSE(£)≈100,796.7
    Epoch 012 | train MSE=0.3

## Results

In [54]:
results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values("RMSE_mean").reset_index(drop=True)
    print("\n=== TOP T-GCN CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===")
    print(results_df.head(10))
    results_df.to_csv("../../results/tgcn_rollingcv_results.csv", index=False)
    print("\nSaved tuning results to ../../results/tgcn_rollingcv_results.csv")
else:
    print("\nNo successful configs to report.")


=== TOP T-GCN CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===
  model_type  WINDOW  HIDDEN_DIM  DROPOUT      LR  WEIGHT_DECAY  folds_used  \
0       TGCN      12         128      0.0  0.0010        0.0000           5   
1       TGCN      12         128      0.0  0.0010        0.0001           5   
2       TGCN      12          64      0.0  0.0010        0.0001           5   
3       TGCN      12          64      0.0  0.0010        0.0000           5   
4       TGCN      12         128      0.3  0.0010        0.0000           5   
5       TGCN      12         128      0.0  0.0005        0.0001           5   
6       TGCN      12         128      0.3  0.0010        0.0001           5   
7       TGCN      12          32      0.0  0.0010        0.0000           5   
8       TGCN      12          64      0.0  0.0005        0.0001           5   
9       TGCN      12          64      0.3  0.0010        0.0000           5   

       MAE_mean       MAE_std     RMSE_mean      RMSE_std  sMAPE_mean  \
